In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#ADLS configuration 
spark.conf.set(
  "fs.azure.account.key.<<Storageaccount_name>>.dfs.core.windows.net",
  dbutils.secrets.get(
    scope="hospitalval",
    key="eventhubconnection"
)
)

bronze_path = "abfss://bronze@hospitalstorageaccoun.dfs.core.windows.net/patient_flow/"
silver_path = "abfss://silver@hospitalstorageaccoun.dfs.core.windows.net/patient_flow_silver"

bronze_df = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

#Defin Schema
schema = StructType([
    StructField("patient_id", StringType()),
    StructField("gender", StringType()),
    StructField("age", IntegerType()),
    StructField("department", StringType()),
    StructField("admission_time", StringType()),
    StructField("discharge_time", StringType()),
    StructField("bed_id", IntegerType()),
    StructField("hospital_id", IntegerType())
])

#Parse it to dataframe
parsed_df = bronze_df.withColumn("data",from_json(col("raw_json"),schema)).select("data.*")

#convert type to Timestamp
clean_df = parsed_df.withColumn("admission_time", to_timestamp("admission_time"))
clean_df = clean_df.withColumn("discharge_time", to_timestamp("discharge_time"))

#invalid admission_times
clean_df = clean_df.withColumn("admission_time",
                               when(
                                   col("admission_time").isNull() | (col("admission_time") > current_timestamp()),
                                   current_timestamp())
                               .otherwise(col("admission_time")))

#Handle Invalid Age
clean_df = clean_df.withColumn("age",
                               when(col("age")>100,floor(rand()*90+1).cast("int"))
                               .otherwise(col("age"))
                               )

#schema evolution
expected_cols = ["patient_id", "gender", "age", "department", "admission_time", "discharge_time", "bed_id", "hospital_id"]

for col_name in expected_cols:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))

#Write to silver table
(
    clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("mergeSchema","true")
    .option("checkpointLocation", silver_path + "_checkpoint")
    .start(silver_path)
)


In [0]:
display(spark.read.format("delta").load(silver_path))

patient_id,gender,age,department,admission_time,discharge_time,bed_id,hospital_id
f0f3172a-3b33-410b-8889-8d80ddc79496,Male,22,Oncology,2026-11-08T17:10:06.318879Z,2026-11-11T08:10:06.318879Z,180,7
7cb8ddc9-d23a-4395-ad4c-f5d37848227b,Male,46,ICU,2026-11-10T13:10:07.322284Z,2026-11-11T20:10:07.322284Z,340,5
a0466e11-2e0b-461b-bbc6-7f1a95bd35e5,Male,2,Oncology,2026-11-10T03:10:08.326219Z,2026-11-12T01:10:08.326219Z,230,6
700f9d3e-b907-4cd1-a8fa-896467be5fec,Female,76,Surgery,2026-11-08T21:10:09.329168Z,2026-11-11T00:10:09.329168Z,97,1
c70819f3-3afe-47ce-bce1-6aa90e5cb2d7,Female,21,Pediatrics,2026-11-08T18:10:10.334658Z,2026-11-11T02:10:10.334658Z,104,6
dc5e6dc9-2c17-402a-8aa1-c596229c1af3,Male,86,Pediatrics,2026-11-10T17:10:11.339316Z,2026-11-12T09:10:11.339316Z,410,7
79bd0c76-366b-4024-937c-492685828d18,Female,53,ICU,2026-11-09T22:10:12.342683Z,2026-11-12T05:10:12.342683Z,450,7
57b89dc3-0331-432a-a776-cd6f51f31e3a,Male,21,ICU,2026-11-09T06:10:13.348177Z,2026-11-09T22:10:13.348177Z,124,3
d2956db9-72d5-47a2-81bb-a7e99b78dd0c,Female,97,Maternity,2026-11-09T07:10:14.354081Z,2026-11-10T10:10:14.354081Z,334,7
6ff426f2-e63b-45ff-adbd-a17e3aad94b1,Male,33,Surgery,2026-11-09T01:10:15.356307Z,2026-11-10T09:10:15.356307Z,476,2
